# 貨幣預測完整流程

本筆記本展示完整的貨幣預測流程：
1. 下載貨幣資料並儲存到 CSV
2. 讀取 CSV 檔案並進行資料處理
3. 切分訓練/驗證/測試集
4. 使用 PatchTST 模型進行訓練
5. 預測未來一週的匯率
6. 視覺化結果與模型表現

## 1. 匯入必要套件與設定

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# 確保可以匯入專案模組
sys.path.append('../src')

# 基本套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import glob
from pathlib import Path

# 機器學習套件
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 專案模組
from currency_predictor.data.collectors import YahooFinanceCollector
from currency_predictor.data.storage import DataStorage
from currency_predictor.data_processor import DataProcessor
from currency_predictor.prediction import CurrencyPredictor, PredictionPipeline

# 🎯 中文字型設定
import matplotlib as mpl
from matplotlib.font_manager import FontProperties, fontManager

# 設定中文字型
chinese_font_path = "D:/tools/iansui/Iansui-Regular.ttf"

if os.path.exists(chinese_font_path):
    print(f"✅ 找到字型檔案: {chinese_font_path}")
    
    chinese_font = FontProperties(fname=chinese_font_path)
    font_name = chinese_font.get_name()
    
    fontManager.addfont(chinese_font_path)
    mpl.rcParams['font.family'] = ['sans-serif']
    mpl.rcParams['font.sans-serif'] = [font_name, 'DejaVu Sans', 'Arial Unicode MS', 'SimHei', 'Microsoft YaHei']
    plt.rcParams['font.family'] = font_name
    mpl.rcParams['axes.unicode_minus'] = False
    
    print(f"✅ 已設定中文字型: {font_name}")
else:
    print(f"❌ 找不到字型檔案，將使用預設字型")

# 設定圖表樣式
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("所有套件匯入成功！")

## 2. 初始化資料收集器並下載貨幣資料

In [ ]:
# 建立資料收集器和儲存管理器
print("📊 初始化資料收集器...")
collector = YahooFinanceCollector()
storage = DataStorage()

print(f"資料儲存路徑: {storage.base_dir}")
print(f"支援的貨幣對: {collector.supported_pairs}")

# 定義目標貨幣對
target_symbols = ['USDTWD=X', 'EURUSD=X', 'EURTWD=X']
print(f"\n目標貨幣對: {target_symbols}")

# 檢查貨幣對可用性
print("\n🔍 檢查貨幣對可用性...")
availability_results = {}
for symbol in target_symbols:
    is_available = collector.check_symbol_availability(symbol)
    availability_results[symbol] = is_available
    status = "✅ 可用" if is_available else "❌ 不可用"
    print(f"  {symbol}: {status}")

available_symbols = [symbol for symbol, available in availability_results.items() if available]
print(f"\n可下載的貨幣對: {available_symbols}")

# 下載並儲存資料
print("\n📥 開始下載並儲存貨幣資料...")
downloaded_data = {}

for symbol in available_symbols:
    print(f"\n正在處理 {symbol}...")
    
    try:
        # 下載一年的資料
        data = collector.get_currency_data(symbol, period="1y", interval="1d")
        
        if data is not None and not data.empty:
            print(f"  ✅ 成功下載 {len(data)} 筆資料")
            print(f"  📅 時間範圍: {data.index.min().strftime('%Y-%m-%d')} 到 {data.index.max().strftime('%Y-%m-%d')}")
            
            # 儲存到本地
            clean_symbol = symbol.replace('=X', '')
            success = storage.save_raw_data(
                data=data,
                symbol=clean_symbol,
                period="1y"
            )
            
            if success:
                print(f"  💾 資料已儲存到本地")
                downloaded_data[symbol] = data
            else:
                print(f"  ❌ 資料儲存失敗")
        else:
            print(f"  ❌ 無法下載 {symbol} 資料")
            
    except Exception as e:
        print(f"  ❌ 處理 {symbol} 時發生錯誤: {str(e)}")

print(f"\n✅ 下載完成！成功下載 {len(downloaded_data)} 個貨幣對")

# 列出已儲存的檔案
print("\n📁 檢查已儲存的資料檔案...")
available_data = storage.list_available_data()
if available_data:
    raw_files = available_data.get('raw_files', [])
    if raw_files:
        print("已儲存的原始資料檔案:")
        for file_info in raw_files:
            print(f"  - {file_info}")

## 3. 讀取 CSV 檔案並進行資料驗證

In [ ]:
# 📊 從 CSV 檔案讀取資料
print("📊 從 CSV 檔案讀取資料...")

# 找出所有 CSV 檔案
csv_files = glob.glob(os.path.join(storage.raw_dir, "*.csv"))
print(f"找到 {len(csv_files)} 個 CSV 檔案")

# 讀取資料並組織
currency_datasets = {}

for csv_file in csv_files:
    try:
        # 從檔名提取貨幣對名稱
        filename = os.path.basename(csv_file)
        symbol_name = filename.split('_')[0]  # 例如從 'USDTWD_1y_20250928_172302.csv' 提取 'USDTWD'
        
        # 讀取 CSV 檔案
        df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
        
        # 處理時區問題
        if hasattr(df.index, 'tz') and df.index.tz is not None:
            df.index = df.index.tz_localize(None)
        
        currency_datasets[symbol_name] = df
        print(f"  ✅ 成功讀取 {symbol_name} 資料，共 {len(df)} 筆")
        print(f"     時間範圍: {df.index.min().strftime('%Y-%m-%d')} 到 {df.index.max().strftime('%Y-%m-%d')}")
        
        # 基本資料檢查
        missing_data = df.isnull().sum()
        if missing_data.sum() > 0:
            print(f"     ⚠️  缺失資料: {missing_data[missing_data > 0].to_dict()}")
        else:
            print(f"     ✅ 無缺失資料")
        
    except Exception as e:
        print(f"  ❌ 讀取 {csv_file} 失敗: {e}")

print(f"\n✅ 成功載入 {len(currency_datasets)} 個貨幣對的資料")

# 選擇主要分析的貨幣對 (選擇 USDTWD 作為主要預測目標)
if 'USDTWD' in currency_datasets:
    main_currency = 'USDTWD'
    main_data = currency_datasets[main_currency]
    print(f"\n🎯 選擇 {main_currency} 作為主要預測目標")
    print(f"資料形狀: {main_data.shape}")
    print(f"欄位: {list(main_data.columns)}")
    
    # 顯示基本統計資訊
    print(f"\n📈 {main_currency} 基本統計:")
    print(main_data.describe())
    
else:
    print("❌ 找不到 USDTWD 資料")
    if currency_datasets:
        main_currency = list(currency_datasets.keys())[0]
        main_data = currency_datasets[main_currency]
        print(f"🔄 改用 {main_currency} 作為主要預測目標")
    else:
        print("❌ 沒有可用的資料")
        main_data = None

## 4. 資料預處理與特徵工程

In [ ]:
if main_data is not None:
    print("🔧 開始資料預處理與特徵工程...")
    
    # 建立資料處理器
    processor = DataProcessor()
    
    # 1. 資料清理
    print("1. 資料清理...")
    clean_data = processor.clean_data(main_data)
    print(f"   清理後資料筆數: {len(clean_data)}")
    
    # 2. 創建技術指標 (使用較小的窗口避免過多 NaN)
    print("2. 創建技術指標...")
    data_with_indicators = processor.create_technical_indicators(clean_data)
    print(f"   技術指標後資料形狀: {data_with_indicators.shape}")
    
    # 3. 創建滯後特徵 (使用較少的滯後期)
    print("3. 創建滯後特徵...")
    processed_data = processor.create_lagged_features(data_with_indicators, lags=[1, 2, 3])
    
    # 檢查處理後的資料
    print(f"   最終處理後資料形狀: {processed_data.shape}")
    print(f"   特徵欄位數量: {len(processed_data.columns)}")
    
    if len(processed_data) == 0:
        print("   ❌ 處理後資料為空，嘗試使用更簡單的特徵...")
        
        # 使用簡化的特徵工程
        simplified_data = clean_data.copy()
        
        # 只添加基本的移動平均和報酬率
        simplified_data['MA_5'] = simplified_data['Close'].rolling(window=5).mean()
        simplified_data['MA_10'] = simplified_data['Close'].rolling(window=10).mean()
        simplified_data['Returns'] = simplified_data['Close'].pct_change()
        simplified_data['High_Low_Ratio'] = simplified_data['High'] / simplified_data['Low']
        simplified_data['Open_Close_Ratio'] = simplified_data['Open'] / simplified_data['Close']
        
        # 移除 NaN 值
        processed_data = simplified_data.dropna()
        print(f"   簡化處理後資料形狀: {processed_data.shape}")
    
    if len(processed_data) > 50:  # 確保有足夠的資料
        print("✅ 資料預處理完成")
        
        # 顯示處理後資料的前幾行
        print(f"\n📊 處理後資料預覽:")
        print(processed_data.head())
        
        # 顯示特徵欄位
        print(f"\n📋 特徵欄位:")
        for i, col in enumerate(processed_data.columns):
            print(f"   {i+1:2d}. {col}")
            
        # 基本統計
        print(f"\n📈 處理後資料統計:")
        print(processed_data.describe())
        
    else:
        print("❌ 處理後資料不足，無法進行模型訓練")
        processed_data = None
        
else:
    print("❌ 無主要資料可供處理")
    processed_data = None

## 5. 訓練/驗證/測試集切分

In [ ]:
if processed_data is not None:
    print("📊 進行時間序列資料切分...")
    
    # 時間序列資料切分 (保持時間順序)
    total_samples = len(processed_data)
    print(f"總樣本數: {total_samples}")
    
    # 切分比例: 70% 訓練, 15% 驗證, 15% 測試
    train_size = int(total_samples * 0.70)
    val_size = int(total_samples * 0.15)
    test_size = total_samples - train_size - val_size
    
    print(f"訓練集: {train_size} 筆 ({train_size/total_samples*100:.1f}%)")
    print(f"驗證集: {val_size} 筆 ({val_size/total_samples*100:.1f}%)")
    print(f"測試集: {test_size} 筆 ({test_size/total_samples*100:.1f}%)")
    
    # 按時間順序切分
    train_data = processed_data.iloc[:train_size]
    val_data = processed_data.iloc[train_size:train_size+val_size]
    test_data = processed_data.iloc[train_size+val_size:]
    
    print(f"\n📅 資料時間範圍:")
    print(f"訓練集: {train_data.index.min().strftime('%Y-%m-%d')} 到 {train_data.index.max().strftime('%Y-%m-%d')}")
    print(f"驗證集: {val_data.index.min().strftime('%Y-%m-%d')} 到 {val_data.index.max().strftime('%Y-%m-%d')}")
    print(f"測試集: {test_data.index.min().strftime('%Y-%m-%d')} 到 {test_data.index.max().strftime('%Y-%m-%d')}")
    
    # 準備特徵和目標變數
    target_column = 'Close'
    feature_columns = [col for col in processed_data.columns if col != target_column]
    
    print(f"\n🎯 目標變數: {target_column}")
    print(f"📊 特徵變數數量: {len(feature_columns)}")
    
    # 分離特徵和目標
    X_train = train_data[feature_columns]
    y_train = train_data[target_column]
    
    X_val = val_data[feature_columns]
    y_val = val_data[target_column]
    
    X_test = test_data[feature_columns]
    y_test = test_data[target_column]
    
    print(f"\n✅ 資料切分完成")
    print(f"X_train 形狀: {X_train.shape}")
    print(f"X_val 形狀: {X_val.shape}")
    print(f"X_test 形狀: {X_test.shape}")
    
    # 檢查資料品質
    print(f"\n🔍 資料品質檢查:")
    print(f"訓練集 NaN 數量: {X_train.isnull().sum().sum()}")
    print(f"驗證集 NaN 數量: {X_val.isnull().sum().sum()}")
    print(f"測試集 NaN 數量: {X_test.isnull().sum().sum()}")
    
    # 顯示目標變數統計
    print(f"\n📊 目標變數統計:")
    print(f"訓練集 {target_column} 範圍: {y_train.min():.4f} ~ {y_train.max():.4f}")
    print(f"驗證集 {target_column} 範圍: {y_val.min():.4f} ~ {y_val.max():.4f}")
    print(f"測試集 {target_column} 範圍: {y_test.min():.4f} ~ {y_test.max():.4f}")
    
else:
    print("❌ 無處理後資料可供切分")

## 6. PatchTST 模型配置與訓練

In [ ]:
if 'X_train' in locals() and X_train is not None:
    print("🤖 配置並訓練 PatchTST 模型...")
    
    try:
        # 建立貨幣預測器
        predictor = CurrencyPredictor(
            model_name="PatchTST",
            model_params={
                'seq_len': 60,      # 使用60天的歷史資料
                'pred_len': 7,      # 預測7天
                'patch_len': 10,    # 每個patch包含10天資料
                'stride': 5,        # patch之間的步長
                'n_estimators': 100, # 隨機森林估計器數量
                'max_depth': 15,    # 最大深度
                'random_state': 42
            }
        )
        
        print("✅ 預測器建立成功")
        print(f"模型參數: {predictor.get_model_info()}")
        
        # 訓練模型
        print("\n🔥 開始訓練模型...")
        
        # 直接使用預測器的模型進行訓練
        start_time = datetime.now()
        predictor.model.fit(X_train, y_train)
        training_time = datetime.now() - start_time
        
        print(f"✅ 模型訓練完成，耗時: {training_time.total_seconds():.2f} 秒")
        
        # 在驗證集上評估
        print("\n📊 驗證集評估...")
        val_predictions = predictor.model.predict(X_val, horizon=1)  # 預測下一天
        
        # 計算評估指標
        # 確保預測值和實際值長度匹配
        if len(val_predictions) > 0:
            # 使用最後N天的實際值進行比較
            val_actual = y_val.iloc[-len(val_predictions):].values
            
            val_mse = mean_squared_error(val_actual, val_predictions)
            val_mae = mean_absolute_error(val_actual, val_predictions)
            val_rmse = np.sqrt(val_mse)
            val_r2 = r2_score(val_actual, val_predictions)
            
            print(f"驗證集 MSE: {val_mse:.6f}")
            print(f"驗證集 MAE: {val_mae:.6f}")
            print(f"驗證集 RMSE: {val_rmse:.6f}")
            print(f"驗證集 R²: {val_r2:.4f}")
            
            # 計算平均絕對百分比誤差 (MAPE)
            val_mape = np.mean(np.abs((val_actual - val_predictions) / val_actual)) * 100
            print(f"驗證集 MAPE: {val_mape:.2f}%")
            
            model_trained = True
        else:
            print("❌ 驗證集預測失敗")
            model_trained = False
            
    except Exception as e:
        print(f"❌ 模型訓練失敗: {str(e)}")
        print(f"錯誤詳情: {e}")
        model_trained = False
        predictor = None
        
else:
    print("❌ 無訓練資料可供模型訓練")

## 7. 預測未來一週的匯率

In [ ]:
if 'model_trained' in locals() and model_trained and predictor is not None:
    print("🔮 預測未來一週的匯率...")
    
    try:
        # 使用最近的資料進行預測
        latest_features = X_test.iloc[-60:]  # 使用最近60天的資料
        
        print(f"使用最近 {len(latest_features)} 天的資料進行預測")
        print(f"最後資料日期: {latest_features.index[-1].strftime('%Y-%m-%d')}")
        
        # 進行7天預測
        future_predictions = predictor.model.predict(latest_features, horizon=7)
        
        print(f"✅ 成功預測未來 {len(future_predictions)} 天")
        
        # 生成未來日期
        last_date = processed_data.index[-1]
        future_dates = [last_date + timedelta(days=i+1) for i in range(7)]
        
        # 建立預測結果資料框
        future_df = pd.DataFrame({
            'Date': future_dates,
            'Predicted_Close': future_predictions
        })
        future_df.set_index('Date', inplace=True)
        
        print("📊 未來一週預測結果:")
        for i, (date, price) in enumerate(future_df.iterrows(), 1):
            print(f"  第{i}天 ({date.strftime('%Y-%m-%d')}): {price['Predicted_Close']:.4f}")
        
        # 計算預測變化
        current_price = processed_data['Close'].iloc[-1]
        prediction_change = future_predictions[-1] - current_price
        prediction_change_pct = (prediction_change / current_price) * 100
        
        print(f"\n📈 預測摘要:")
        print(f"當前價格: {current_price:.4f}")
        print(f"一週後預測價格: {future_predictions[-1]:.4f}")
        print(f"預期變化: {prediction_change:+.4f} ({prediction_change_pct:+.2f}%)")
        
        # 在測試集上評估模型表現
        print(f"\n📊 測試集評估...")
        test_predictions = predictor.model.predict(X_test, horizon=1)
        
        if len(test_predictions) > 0:
            test_actual = y_test.iloc[-len(test_predictions):].values
            
            test_mse = mean_squared_error(test_actual, test_predictions)
            test_mae = mean_absolute_error(test_actual, test_predictions)
            test_rmse = np.sqrt(test_mse)
            test_r2 = r2_score(test_actual, test_predictions)
            test_mape = np.mean(np.abs((test_actual - test_predictions) / test_actual)) * 100
            
            print(f"測試集 MSE: {test_mse:.6f}")
            print(f"測試集 MAE: {test_mae:.6f}")
            print(f"測試集 RMSE: {test_rmse:.6f}")
            print(f"測試集 R²: {test_r2:.4f}")
            print(f"測試集 MAPE: {test_mape:.2f}%")
            
            # 儲存預測結果供視覺化使用
            prediction_results = {
                'future_predictions': future_df,
                'test_predictions': test_predictions,
                'test_actual': test_actual,
                'test_dates': y_test.iloc[-len(test_predictions):].index,
                'current_price': current_price,
                'metrics': {
                    'test_mse': test_mse,
                    'test_mae': test_mae,
                    'test_rmse': test_rmse,
                    'test_r2': test_r2,
                    'test_mape': test_mape
                }
            }
            
            prediction_success = True
        else:
            print("❌ 測試集預測失敗")
            prediction_success = False
            
    except Exception as e:
        print(f"❌ 預測失敗: {str(e)}")
        prediction_success = False
        
else:
    print("❌ 模型未訓練成功，無法進行預測")

## 8. 視覺化結果與模型表現